# XRD: Bruker DIFFRAC.EVA `.xy`

`XRDData` holds a diffractogram over the 2θ angle in degrees. The anode, and so
the wavelength, comes from the header line of the export. The files are the test
fixtures under `tests/xrd/`: a MoO3/TiO2 pattern and two ICDD reference cards.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
from lab_data_parser.xrd.pdf_card import ReferenceCard

XRD = Path("..") / "tests" / "xrd"
xrd = ldp.XRDData.from_bruker_xy(XRD / "bruker_MoO3-12pct-TiO2_12-28deg.xy")
print(xrd.metadata["anode"], xrd.da.attrs["wavelength_angstrom"], "angstrom")
print(xrd.angle.min(), "to", xrd.angle.max(), "deg")

## Reference cards

ICDD cards exported as text; `two_theta` recomputes the line positions from the
d-spacings for the wavelength of the measurement.

In [ ]:
cards = {
    r"anatase $\mathsf{TiO_2}$": ReferenceCard.from_file(XRD / "PDF 78-2486_TiO2-Anatase.txt"),
    r"$\mathsf{MoO_3}$": ReferenceCard.from_file(XRD / "PDF 35-0609_MoO3.txt"),
}
wavelength = xrd.da.attrs["wavelength_angstrom"]

fig, ax = plt.subplots(figsize=(7, 4))
smoothed = xrd.smooth_savgol()
ax.plot(smoothed.angle, smoothed.values, color="k", lw=0.8)
top = np.nanmax(smoothed.values)
for color, (label, card) in zip(("C0", "C1"), cards.items()):
    positions = card.two_theta(wavelength)
    inside = (positions > xrd.angle.min()) & (positions < xrd.angle.max())
    ax.vlines(positions[inside], 0, top * card.intensity[inside] / card.intensity.max(), color=color, label=label)
ax.set_xlabel("2θ / deg")
ax.set_ylabel("intensity / counts")
ax.legend()
plt.show()

## Fit one reflection and estimate the crystallite size

`fit_reflection` fits a pseudo-Voigt plus a linear background in a window,
`scherrer_size` gives the crystallite size in nm (the instrumental width is
subtracted in quadrature).

In [ ]:
fit = xrd.fit_reflection(25.3, window_deg=1.5)          # anatase (101)
size_nm = xrd.scherrer_size(fit.position_deg, fit.fwhm_deg, instrumental_fwhm_deg=0.08)
print(f"position {fit.position_deg:.3f} deg, fwhm {fit.fwhm_deg:.3f} deg, size {size_nm:.1f} nm")

In [ ]:
window = xrd.select_angle_range(*fit.window_deg)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(window.angle, window.values, "k.", ms=3, label="data")
ax.plot(window.angle, fit.model(window.angle), label="fit")
ax.set_xlabel("2θ / deg")
ax.set_ylabel("intensity / counts")
ax.legend()
plt.show()

## Other axes

`d_spacing` and `q_vector` take the wavelength in angstrom.

In [ ]:
print(xrd.d_spacing(wavelength)[[0, -1]], "angstrom")
print(xrd.q_vector(wavelength)[[0, -1]], "angstrom^-1")